# DRISHTI — cloud-T4 runner (Colab / Kaggle)

Runs the **GPU-heavy** stages of the DRISHTI offline pipeline on a free T4 session, then lets you
download the resulting bundle to finish and serve on the ground (the CPU tier).

**This notebook is honest about what it does.** It installs the *same* package as the local tier,
runs the *real* CLI, and reports the *real* doctor output — no stubs, no fabricated results. If a
dependency or the GPU is missing, the doctor and the run will say so and fail loudly.

**Tier split** (see `docs/implementation/02-SYSTEM-DESIGN.md`): the neural stages
`s2_poses`, `s3_masking`, `s4_depth`, `s7_dense` run here on CUDA; everything else
(`s0`, `s1`, `s6`, `s8`, `s9`, `s10`, `report`) runs on the CPU ground station. The bundle is
resumable and content-hashed, so stages completed here are skipped when you resume locally.

### How to use
1. **Runtime → Change runtime type → GPU (T4)** in Colab, or enable the GPU accelerator on Kaggle.
2. Set `REPO_URL` (and `REPO_REF`) below to your DRISHTI repository.
3. Run the cells top to bottom. Upload your mission (video + telemetry + `mission.yaml`) when prompted.
4. Download the zipped bundle at the end.

## 0 · Confirm the GPU

If this errors or shows no Tesla T4, stop and enable the GPU runtime — the heavy stages need it.

In [ ]:
!nvidia-smi

## 1 · Get the code

Point these at your repository. On Kaggle you may instead attach the repo as a dataset and set
`REPO_DIR` to its path (e.g. `/kaggle/input/drishti`) — then skip the clone.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/<your-org>/drishti.git"  # <-- set me
REPO_REF = "main"                                        # branch, tag, or commit
REPO_DIR = Path("/content/drishti") if Path("/content").exists() else Path("/kaggle/working/drishti")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"{REPO_DIR} already present — skipping clone")

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

## 2 · Install DRISHTI (GPU extras, auto-handles Colab's Python 3.13)

Installs the package with the heavy extras. Two honesty notes baked into the cell:

- **Open3D needs Python ≤ 3.12.** `s7_dense` (TSDF) uses Open3D, which has no wheel past cp312. Current
  Colab / Kaggle run **Python 3.13**, so if the kernel is newer the cell builds an isolated **Python
  3.12** env with [`uv`](https://github.com/astral-sh/uv), installs everything there, and prepends it to
  `PATH` — every later `!drishti` / `!python` uses it. On a ≤ 3.12 kernel it installs in place and keeps
  the hosted CUDA PyTorch.
- **`transformers` isn't in any extra**, but S3 (RT-DETR) and S4 (Depth-Anything-V2) load their models
  through it — so it's installed explicitly, or `--upto s7_dense` would block at S3/S4.

Each group installs in isolation (one bad wheel can't take down the CLI). The cell ends with an
`OK`/`MISSING` capability probe — the same imports §3 `doctor` gates each stage on.

In [ ]:
# Install the DRISHTI stack for the GPU-heavy stages. Two things this handles honestly:
#   (1) Open3D (needed by s7_dense TSDF) ships wheels only through cp312, but Colab/Kaggle now run
#       Python 3.13 -> if the kernel is newer, build a uv-managed Python 3.12 env, install EVERYTHING
#       there, and prepend it to PATH so every later cell (!drishti, !python) uses it.
#   (2) transformers is NOT in any pyproject extra, yet s3_masking (RT-DETR) and s4_depth
#       (Depth-Anything-V2) load their models through it -> install it explicitly, or those stages block.
# Each group installs in isolation, so one un-buildable wheel can't take down the `drishti` CLI.
import os, sys, subprocess
from pathlib import Path

def sh(cmd):
    print("  $", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    return subprocess.run(cmd, shell=isinstance(cmd, str)).returncode

pyver = sys.version_info[:2]
USE_VENV = pyver > (3, 12)   # open3d has no wheel past cp312
print(f"kernel Python {pyver[0]}.{pyver[1]}  ->  "
      f"{'building an isolated Python 3.12 env (open3d has no wheel here)' if USE_VENV else 'installing in place'}\n")

if not USE_VENV:
    PY = sys.executable
else:
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
    VENV = (Path("/content") if Path("/content").exists() else Path("/kaggle/working")) / "venv312"
    sh(["uv", "python", "install", "3.12"])
    if not (VENV / "bin" / "python").exists():
        sh(["uv", "venv", "--seed", "--python", "3.12", str(VENV)])   # --seed => venv gets pip, so later !pip works too
    bindir = VENV / "bin"
    os.environ["VIRTUAL_ENV"] = str(VENV)
    os.environ["PATH"] = f"{bindir}{os.pathsep}{os.environ['PATH']}"   # !drishti / !python / subprocess -> this venv
    PY = str(bindir / "python")
    print("\nusing", PY, "\n")

def pipi(*pkgs, required=False):
    """Install into the target interpreter. Isolated: one failing native wheel can't abort the rest."""
    cmd = (["uv", "pip", "install", "--python", PY] if USE_VENV else [PY, "-m", "pip", "install", "-q"]) + list(pkgs)
    if sh(cmd) != 0:
        if required:
            raise RuntimeError(f"REQUIRED install failed: {' '.join(pkgs)} — cannot continue.")
        print(f"  ! optional install failed: {' '.join(pkgs)} — §3 `doctor` will show what's blocked\n")

# A fresh 3.12 venv has no torch; the hosted (>=3.13) kernel already ships the right CUDA torch, so keep that.
if USE_VENV:
    pipi("torch")   # PyPI Linux torch is the CUDA build; it uses the host's NVIDIA driver
subprocess.run([PY, "-c", "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"])

# DRISHTI core first (guarantees the CLI), then each heavy capability group in isolation.
pipi("-e", ".", required=True)
for grp in ("video", "geo", "recon", "poses", "spine", "telem", "depth", "server"):
    pipi("-e", f".[{grp}]")

# Packaging gaps: transformers (S3/S4 models) is required for --upto s7_dense; laspy/pillow keep parity
# with the full notebook (LAS runs on the ground station, but installing here is harmless).
pipi("-U", "transformers>=4.45", "timm", "safetensors")
pipi("laspy[lazrs]>=2.5")
pipi("pillow")

# Honest capability probe — the same imports §3 `doctor` gates each stage on.
probe = (
    "import importlib.util as u\n"
    "mods=['av','cv2','pyproj','rasterio','shapely','open3d','trimesh','pycolmap','gtsam','torch','transformers','laspy']\n"
    "print('\\ninstalled capabilities:')\n"
    "for m in mods:\n"
    "    print('  %-13s %s' % (m, 'OK' if u.find_spec(m) else 'MISSING'))\n"
)
subprocess.run([PY, "-c", probe])

## 3 · Doctor — verify real capabilities before spending GPU minutes

Expect `cuda: true` and `cuda_name: "Tesla T4"`. The `stages` map shows which stages are ready in
*this* environment.

In [ ]:
!drishti doctor

## 4 · Provide the mission inputs

A DRISHTI mission is a `mission.yaml` descriptor plus its referenced files (the drone video, the GPS/
telemetry track, optional IMU/baro/intrinsics). Pick ONE source below.

- **Generate one here — Option D (recommended for a first real run):** no upload needed. Assemble a
  realistic dataset directly on this T4 — either **real** open aerial imagery + its **real GPS EXIF**, or
  a **ground-truth synthetic** city. This is the best place to fetch the large `aukerman` set (real
  **buildings**, ~543 MB), since the full run happens right here.
- **Colab + Google Drive** (best for your own large video): mount Drive and set `MISSION` to the descriptor.
- **Direct upload** (small clips): use the upload widget.
- **Kaggle dataset**: attach it and set `MISSION` to its path under `/kaggle/input/...`.

### Option D · generate a realistic dataset here

Runs `scripts/make_sample_dataset.py` (see [`scripts/README.md`](../scripts/README.md)) on this T4 and
points `DRISHTI_MISSION` at the generated descriptor, so the options cell and the run cell below use it
automatically. `aukerman` gives real **buildings** (real pixels + real GPS EXIF); `synthetic` gives a
ground-truth city whose known geometry makes accuracy *checkable*. For the real sets, only real open
imagery and its own GPS are used — nothing about the world is fabricated (only playback timing is).

In [ ]:
# --- Option D: generate a realistic dataset ON THIS T4 (no upload needed) ---
DATASET = "aukerman"   # aukerman | brighton_beach | caliterra | lewis | synthetic

!pip install -q pillow  # EXIF reader for the real path; opencv ships with the video extra

if DATASET == "synthetic":
    !python scripts/make_sample_dataset.py --synthetic
    _name = "synthetic_city"
else:
    !python scripts/make_sample_dataset.py --dataset {DATASET}
    _name = DATASET

MISSION = str(REPO_DIR / "configs" / "datasets" / f"{_name}.yaml")
os.environ["DRISHTI_MISSION"] = MISSION   # the options cell below reads this if you skip A/B/C
print("\nDRISHTI_MISSION =", MISSION)
!cat "{MISSION}"

In [ ]:
# --- Option A: Google Drive (Colab) ---
# from google.colab import drive; drive.mount('/content/drive')
# MISSION = '/content/drive/MyDrive/drishti/mission.yaml'

# --- Option B: direct upload (Colab) ---
# from google.colab import files; up = files.upload()   # upload mission.yaml + video + telemetry
# MISSION = next(iter(up))

# --- Option C: Kaggle dataset / already-present path ---
MISSION = os.environ.get("DRISHTI_MISSION", "/content/drive/MyDrive/drishti/mission.yaml")  # <-- set me

assert Path(MISSION).is_file(), f"mission descriptor not found: {MISSION}"
print("mission:", MISSION)

## 5 · Run the GPU-heavy stages

`--upto s7_dense` runs the pipeline through dense reconstruction — i.e. all the stages that benefit
from the T4 — and stops. The remaining CPU stages (`s8_mesh` → `report`) are meant to run on the
ground station.

We pass `--profile max` for the highest-quality settings the T4 can hold, but override
`dense.method` / `mesh.method` back to the implemented **TSDF + Poisson** path. `max` otherwise selects
`gaussian` / `2dgs`, which this build does **not** implement and which **raise loudly** at `s7_dense` /
`s8_mesh` — and because the later local `resume` reuses the config recorded in the bundle, the override
also keeps that `s8_mesh` step working.

The bundle is written under `RUNS_DIR`. Everything the runner does is recorded in `manifest.json`
(per-stage status, timings, confidence, and `degraded_reason` if a fallback was used).

In [ ]:
RUNS_DIR = str(REPO_DIR / "runs")
os.environ["DRISHTI_RUNS_DIR"] = RUNS_DIR

# Full quality, GPU stages only. Drop --upto to run the ENTIRE pipeline here instead.
# NOTE 1: `run` takes the descriptor via the required --dataset/-d option (there is no positional form).
# NOTE 2: `max` selects dense.method=gaussian + mesh.method=2dgs, which this build does NOT implement
#         (s7_dense / s8_mesh raise loudly). We keep max's other quality knobs but override those two to
#         the implemented TSDF + Poisson path, so this run — and the later local `resume` of s8 — succeed.
!drishti run --dataset "$MISSION" --profile max --set dense.method=tsdf --set mesh.method=poisson --upto s7_dense

# Show what was produced.
!drishti stages
print("\nBundles in", RUNS_DIR, ":")
for d in sorted(Path(RUNS_DIR).glob("*/manifest.json")):
    print(" ", d.parent.name)

## 6 · Inspect the run (honest status)

`inspect` prints the manifest: which stages ran, where (`local` / `cloud-t4`), how long, the measured
confidence per stage, and any degradation. This is the ground truth — not a progress bar.

In [ ]:
# Inspect the most recent bundle.
bundles = sorted(Path(RUNS_DIR).glob("*/manifest.json"), key=lambda p: p.stat().st_mtime)
assert bundles, "no bundle was produced — check the run cell output above"
LATEST = bundles[-1].parent
!drishti inspect "$LATEST"
!drishti verify "$LATEST"   # recompute output hashes vs. the manifest (integrity check)

## 7 · Download the bundle

Zip the bundle and pull it to your machine, then **resume** on the ground station. `resume` reuses the
config recorded in the bundle's manifest (so it stays reproducible) and skips the GPU stages already
marked done, picking up at `s8_mesh`:

```bash
unzip <run_id>.zip -d runs/
drishti resume runs/<run_id>      # continues at s8_mesh; the GPU stages are already 'done'
```

In [ ]:
import shutil

archive = shutil.make_archive(str(LATEST), "zip", root_dir=LATEST.parent, base_dir=LATEST.name)
print("wrote", archive, f"({Path(archive).stat().st_size / 1e6:.1f} MB)")

try:
    from google.colab import files  # type: ignore
    files.download(archive)
except Exception:
    print("Not on Colab — download", archive, "from the file browser (Kaggle: /kaggle/working).")